# Data preprocessing for master-oez61

In [126]:
!rm -r data/
!rm -r __MACOSX/

rm: __MACOSX/: No such file or directory


In [127]:
!mkdir data
!unzip data.zip -d ./data

Archive:  data.zip
  inflating: ./data/README.dataset.txt  
  inflating: ./data/README.roboflow.txt  
  inflating: ./data/data.yaml        
   creating: ./data/test
   creating: ./data/test/images
 extracting: ./data/test/images/08-3_jpg.rf.1d2e1fd59a8062bd65c08da9a0c6d1a6.jpg  
 extracting: ./data/test/images/10-3_jpg.rf.1019a21ed314ac7bf6026b13060b59ca.jpg  
 extracting: ./data/test/images/202205313_jpg.rf.38f861fa43c15bf4bef7fc54e8746a6f.jpg  
 extracting: ./data/test/images/202205313_jpg.rf.feaeefb01da6def2c0408fe59ba72359.jpg  
 extracting: ./data/test/images/202205314_jpg.rf.7189979dee8259ed2ee73f64ac806669.jpg  
 extracting: ./data/test/images/202205315_jpg.rf.4d58f9e123687224c78149c711bc8bd8.jpg  
 extracting: ./data/test/images/202205315_jpg.rf.ff7f48a68dddc439c14678207df3515d.jpg  
 extracting: ./data/test/images/202205316_jpg.rf.1ea6c8da619fbe63cce6b8f712a9c38f.jpg  
 extracting: ./data/test/images/202205316_jpg.rf.ce237eda3d5e98fbb717a7b912cf078b.jpg  
 extracting: ./data/t

In [128]:
import yaml
import os

# Modify original data label
def update_original_data(yaml_path, labels_paths):
    yaml_data = {}
    with open(yaml_path, 'r') as f:
        yaml_data = yaml.safe_load(f)

    labels = yaml_data['names']
    for path in labels_paths:
        for file in os.listdir(path):
            new_content = []
            # Read label files
            with open(f'{path}/{file}', 'r') as f:
                content = f.readlines()
                for line in content:
                    # Skip 1s and 5z, as this dataset is using richii mahjong
                    idx = int(line.split()[0])
                    if labels[idx] in ['1s', '5z']:
                        continue

                    # update the class id to 0, as only 1 class
                    words = line.split(' ')[1:]
                    new_words = ['0'] + words
                    new_line = ' '.join(new_words)
                    new_content.append(new_line)


            # Write label files
            with open(f'{path}/{file}', 'w') as f:
                f.writelines(new_content)
                f.close()

In [129]:
from os.path import split
import os

# Filter out malformatted files
# For yolo without OBB, each line in label file contains only 5 values
def malformat_filter(paths):
    for path in paths:
        targets = []
        for file in os.listdir(path+'labels'):
            lines = []
            label_path = path+'labels/'+file
            with open(label_path, 'r') as f:
                lines = f.readlines()

            for line in lines:
                words = line.split(' ')
                if len(words) > 5:
                    targets.append('.'.join(label_path.split('.')[:-1]))
                    break

        for target in targets:
            os.remove(target+'.txt')
            os.remove(target.replace('labels', 'images', 1)+'.jpg')

In [130]:
# Modify original yaml to accomodate Hong Kong Mahjong Tiles
import yaml
import os

def update_original_yaml(path):
    yaml_content = {}
    with open(path, 'r') as f:
        yaml_content = yaml.safe_load(f)
        print(yaml_content)
        f.close()

    yaml_content['path'] = './data'
    yaml_content['test'] = 'test/images'
    yaml_content['train'] = 'train/images'
    yaml_content['val'] = 'valid/images'
    yaml_content['names'] = ['tile']
    yaml_content['nc'] = len(yaml_content['names'])

    with open('yolo11.yaml', 'w') as f:
        yaml.safe_dump(yaml_content, f)

In [131]:
# Update data/train/labels
labels_paths = ['data/train/labels', 'data/valid/labels']
update_original_data('data/data.yaml', labels_paths)
malformat_filter(['data/train/', 'data/valid/', 'data/test/'])

# Data preprocessing for mahjong-tiles-non-riichi-xe0ya
- Copies '1s' and '5z' to main dataset 'data/'

In [132]:
!rm -r data1/
!rm -r __MACOSX/

rm: __MACOSX/: No such file or directory


In [133]:
!mkdir data1/
!unzip data_cls.zip -d data1/

Archive:  data_cls.zip
   creating: data1//valid
   creating: data1//valid/images
  inflating: data1//valid/images/367362342_966412384458625_1506370883166840771_n_jpg.rf.d3e8cdee9581483db48c5d8af709f7ff.jpg  
  inflating: data1//valid/images/368073213_1367789004123469_2851708390906449067_n_jpg.rf.4a640e92c567146f8ea4e29b5a6325d9.jpg  
  inflating: data1//valid/images/368097815_362001246166410_9137931599732096843_n_jpg.rf.7637eafa7f34c7260bb37ce7d8c97da6.jpg  
  inflating: data1//valid/images/370080665_351299340801258_8710629831104354977_n_jpg.rf.c31ffb0c2e669a4e4a714d9ef740b6bd.jpg  
  inflating: data1//valid/images/370211613_719383850071706_5882000634833572319_n_jpg.rf.67ca39c412ef09eca5b65b4790fa1bc7.jpg  
  inflating: data1//valid/images/370225297_240919035658149_8182226098493293914_n_jpg.rf.f51024a3631b5d5a97fc9776ecf82a7d.jpg  
  inflating: data1//valid/images/370228378_1490243705105555_5117975651034644069_n_jpg.rf.67cb87f47a939c05cbe7c4e7d20f2462.jpg  
  inflating: data1//valid/i

In [134]:
# Traverse all train + val (image, label) pairs
# If a line in label file is in ['1s', '5z'], update to 'tile' and copy it to 'data'

import shutil
import os
def update_data_copy(path, target):
    train_path = os.path.join(path, 'train')
    valid_path = os.path.join(path, 'valid')
    
    yaml_data = {}
    with open(os.path.join(path, 'data.yaml') , 'r') as f:
        yaml_data = yaml.safe_load(f)

    labels = yaml_data['names']

    # Traverse all label files in train
    train_label_path = os.path.join(train_path, 'labels/')
    for label_file in os.listdir(train_label_path):
        file_content = []
        new_content = []
        with open(os.path.join(train_label_path, label_file), 'r') as f:
            file_content = f.readlines()

        for line in file_content:
            idx = int(line.split()[0])
            label_name = labels[idx]
            if label_name in ['s1', 'z5']:
                words = line.split()[1:]
                words = ['0'] + words
                new_content.append(words)

        if new_content:
            text_content = []
            for line in new_content:
                text_content.append(' '.join(line) + '\n')
            
            with open(os.path.join(train_label_path, label_file), 'w') as f:
                f.writelines(text_content)
            
            # Copy to data
            train_image_path = os.path.join(train_path, 'images')
            img_path = os.path.join(train_image_path, label_file[:-4] + '.jpg')
            print(img_path)
        else: 
            # remove those files
            train_image_path = os.path.join(train_path, 'images')
            os.remove(os.path.join(train_image_path, label_file[:-4] + '.jpg'))
            os.remove(os.path.join(train_label_path, label_file))
    
    # Copy all path/train/(images, labels) to target/train/(image, labels)
    train_image_path = os.path.join(train_path, 'images/')
    os.makedirs(os.path.join(target, 'train', 'images'), exist_ok=True)
    os.makedirs(os.path.join(target, 'train', 'labels'), exist_ok=True)
    for f in os.listdir(train_image_path):
        shutil.copy2(os.path.join(train_image_path, f), os.path.join(target, 'train', 'images'))

    for f in os.listdir(train_label_path):
        shutil.copy2(os.path.join(train_label_path, f), os.path.join(target, 'train', 'labels'))


    # Traverse all label files in valid
    valid_label_path = os.path.join(valid_path, 'labels')
    for label_file in os.listdir(valid_label_path):
        file_content = []
        new_content = []
        with open(os.path.join(valid_label_path, label_file), 'r') as f:
            file_content = f.readlines()

        for line in file_content:
            idx = int(line.split()[0])
            label_name = labels[idx]
            if label_name in ['s1', 'z5']:
                words = line.split()[1:]
                words = ['0'] + words
                new_content.append(words)

        if new_content:
            text_content = []
            for line in new_content:
                text_content.append(' '.join(line) + '\n')
            
            with open(os.path.join(valid_label_path, label_file), 'w') as f:
                f.writelines(text_content)
            
            # Copy to data
            valid_image_path = os.path.join(valid_path, 'images')
            img_path = os.path.join(valid_image_path, label_file[:-4] + '.jpg')
            print(img_path)
        else: 
            # remove those files
            valid_image_path = os.path.join(valid_path, 'images')
            os.remove(os.path.join(valid_image_path, label_file[:-4] + '.jpg'))
            os.remove(os.path.join(valid_label_path, label_file))

    # Copy all path/train/(images, labels) to target/train/(image, labels)
    valid_image_path = os.path.join(valid_path, 'images/')
    os.makedirs(os.path.join(target, 'valid', 'images'), exist_ok=True)
    os.makedirs(os.path.join(target, 'valid', 'labels'), exist_ok=True)
    for f in os.listdir(valid_image_path):
        shutil.copy2(os.path.join(valid_image_path, f), os.path.join(target, 'valid', 'images'))

    for f in os.listdir(valid_label_path):
        shutil.copy2(os.path.join(valid_label_path, f), os.path.join(target, 'valid', 'labels'))


In [135]:
malformat_filter(['data1/train/', 'data1/valid/', 'data1/test/'])
update_data_copy('./data1', './data')

./data1/train/images/hand_155_jpg.rf.9e00a59f8a306c1d0dbcf51f56b260dd.jpg
./data1/train/images/395523960_1047983586655468_3617836938143150326_n_jpg.rf.16f5aa6769e424f349d37318f7a63bba.jpg
./data1/train/images/385360810_694805725914528_189238597542184190_n_jpg.rf.708f8cf2d62e5b4e4baff3e948598fb9.jpg
./data1/train/images/394632172_220929894204758_5130022221510297090_n_jpg.rf.38b697702cf64d4736a6ecf39001fbe7.jpg
./data1/train/images/hand_011_jpg.rf.74cfac47ac8c825415cba3b21d9a959e.jpg
./data1/train/images/V7Aa5mF0SGP2Uf3KeWCRVoUHxsx2MS_jpg.rf.68a1109b83f0033ab16aed3ebd305ffa.jpg
./data1/train/images/396063485_293427056846610_4291922774061432571_n_jpg.rf.d9a016fb1fc0641884d7393b5090bb6a.jpg
./data1/train/images/403791264_902504744730573_7512458673612944762_n_jpg.rf.035d07049318d031de5c56c5c2df2bbc.jpg
./data1/train/images/395412092_187983820948794_9166548803510853661_n_jpg.rf.f0a397902a8765235b037384268eca45.jpg
./data1/train/images/411129254_295957009679121_4484421262531516219_n_jpg.rf.df

In [137]:
update_original_yaml('./data/data.yaml')

{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 38, 'names': ['0b', '0m', '0p', '0s', '1m', '1p', '1s', '1z', '2m', '2p', '2s', '2z', '3m', '3p', '3s', '3z', '4m', '4p', '4s', '4z', '5m', '5p', '5s', '5z', '6m', '6p', '6s', '6z', '7m', '7p', '7s', '7z', '8m', '8p', '8s', '9m', '9p', '9s'], 'roboflow': {'workspace': 'majmaster', 'project': 'master-oez61', 'version': 7, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/majmaster/master-oez61/dataset/7'}}
